# Differentially Private SVM - Gallstone Dataset\n\nImplements a custom Differentially Private SVM using Huber loss with objective perturbation (Chaudhuri et al. 2011).


## 1. Import Libraries


In [1]:
import time
import os
import sys
import json
import pickle
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

warnings.filterwarnings('ignore')

# The DP-SVM implementation and the model exporter are shared across all
# datasets and live at the repository root.
REPO_ROOT = os.path.abspath(os.path.join('..', '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from common_svm import DifferentiallyPrivateSVM  # noqa: E402
from model_exporter import export_model  # noqa: E402

# Shared metric collector (repository root): logs training accuracy,
# balanced accuracy and AUROC alongside the metrics already tracked below.
import os as _os
import sys as _sys
_REPO_ROOT = _os.path.abspath(_os.path.join('..', '..'))
if _REPO_ROOT not in _sys.path:
    _sys.path.insert(0, _REPO_ROOT)
from metrics_utils import ExtraMetrics


## 2. Load Preprocessed Data


In [2]:
with open('../data/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']

# DP norm precondition: clip every row to unit L2 norm (||x_i||_2 <= 1).
X_train = X_train / np.maximum(np.linalg.norm(X_train, axis=1, keepdims=True), 1.0)
X_test = X_test / np.maximum(np.linalg.norm(X_test, axis=1, keepdims=True), 1.0)

# Convert labels to {-1, +1} for SVM objective perturbation
y_train_svm = 2 * y_train - 1
y_test_svm = 2 * y_test - 1


## 3. Load Baseline Results


In [3]:
try:
    baseline_df = pd.read_csv('output/baseline_results.csv')
    baseline_accuracy = baseline_df['accuracy'].values[0]
    baseline_f1 = baseline_df['f1_score'].values[0]
except FileNotFoundError:
    baseline_accuracy = None
    baseline_f1 = None


## 4. DP-SVM Implementation

**Implementation:** `DifferentiallyPrivateSVM` from `common_svm.py` at the repository root — a single shared implementation of Algorithm 2 (Chaudhuri, Monteleoni & Sarwate, JMLR 2011) used by every dataset.

`normalize=True` rescales the features by the largest training-row norm so the privacy precondition ‖x_i‖ ≤ 1 holds. The scale is folded back into the learned weights, so predictions (and the exported models) still take the unscaled features.

`fit_intercept=True` (the default) learns an unregularized bias term; the noise vector is drawn in d = n_features + 1 dimensions accordingly.

`privacy_report()` records ε′, Δ and whether Algorithm 2's fallback branch fired. When it does, the effective budget is ε/2 rather than ε — the sweep below stores this per ε in the JSON report.


## 5. Sweep Epsilon


In [4]:
epsilon_values = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 4.0, 6.0, 8.0, 10.0]
N_RUNS = 30

results = {
    'epsilon': [], 'accuracy_mean': [], 'accuracy_std': [],
    'f1_score_mean': [], 'f1_score_std': [],
    'precision_mean': [], 'precision_std': [],
    'recall_mean': [], 'recall_std': [],
}

all_results = {
    'model_type': 'Differentially Private SVM (Huber Loss + Objective Perturbation, Algorithm 2)',
    'dataset': 'Gallstone',
    'n_runs_per_epsilon': N_RUNS,
    'epsilon_results': {}
}

print(f"Training DP SVM with {N_RUNS} runs per epsilon...")
print('=' * 80)

for eps in epsilon_values:
    run_accs, run_f1s, run_precs, run_recs, run_cms = [], [], [], [], []

    extra = ExtraMetrics()
    for run in range(N_RUNS):
        seed = run * 10 + 42
        dp_svm = DifferentiallyPrivateSVM(epsilon=eps, Lambda=0.01, h=0.5,
                                          normalize=True, random_state=seed)
        _t0 = time.perf_counter()
        dp_svm.fit(X_train, y_train_svm)
        train_time = time.perf_counter() - _t0
        y_pred_svm = dp_svm.predict(X_test)
        
        # Map labels back to {0, 1}
        y_pred = (y_pred_svm + 1) // 2
        
        run_accs.append(accuracy_score(y_test, y_pred))
        extra.add(y_test, y_pred, model=dp_svm, X=X_test, train_time=train_time)
        run_f1s.append(f1_score(y_test, y_pred, zero_division=0))
        run_precs.append(precision_score(y_test, y_pred, zero_division=0))
        run_recs.append(recall_score(y_test, y_pred, zero_division=0))
        run_cms.append(confusion_matrix(y_test, y_pred).tolist())
        if run == 0:
            export_model(dp_svm, f'output/model/dp_svm_model_eps_{eps}.pkl')
            privacy = dp_svm.privacy_report()

    acc_mean, acc_std = np.mean(run_accs), np.std(run_accs)
    f1_mean, f1_std = np.mean(run_f1s), np.std(run_f1s)
    prec_mean, prec_std = np.mean(run_precs), np.std(run_precs)
    rec_mean, rec_std = np.mean(run_recs), np.std(run_recs)

    results['epsilon'].append(eps)
    # Merge in the added metrics; keys the notebook already tracks are skipped.
    for _key, _value in extra.summary().items():
        results.setdefault(_key, []).append(_value)
    results['accuracy_mean'].append(acc_mean)
    results['accuracy_std'].append(acc_std)
    results['f1_score_mean'].append(f1_mean)
    results['f1_score_std'].append(f1_std)
    results['precision_mean'].append(prec_mean)
    results['precision_std'].append(prec_std)
    results['recall_mean'].append(rec_mean)
    results['recall_std'].append(rec_std)

    all_results['epsilon_results'][str(eps)] = {
        'epsilon': float(eps), 'n_runs': N_RUNS,
        'accuracy_mean': acc_mean, 'accuracy_std': acc_std,
        'f1_mean': f1_mean, 'f1_std': f1_std,
        'precision_mean': prec_mean, 'precision_std': prec_std,
        'recall_mean': rec_mean, 'recall_std': rec_std,
        'per_run_accuracies': run_accs, 'per_run_f1s': run_f1s,
        'per_run_confusion_matrices': run_cms,
        'privacy': privacy
    }
    all_results['epsilon_results'][str(eps)].update(extra.summary())
    all_results['epsilon_results'][str(eps)].update(extra.per_run())

    print(f"eps={eps:5.2f} | Acc: {acc_mean:.4f}+/-{acc_std:.4f} | F1: {f1_mean:.4f}+/-{f1_std:.4f}")

    if privacy['fallback_triggered']:
        print(f"        Algorithm 2 fallback fired: effective \u03b5' ="
              f" {privacy['eps_prime']:.4f}, \u0394 = {privacy['Delta']:.6f}")

print('=' * 80)


Training DP SVM with 30 runs per epsilon...


Model successfully exported to output/model/dp_svm_model_eps_0.1.pkl
eps= 0.10 | Acc: 0.5026+/-0.0318 | F1: 0.3182+/-0.2802
        Algorithm 2 fallback fired: effective ε' = 0.0500, Δ = 0.144910
Model successfully exported to output/model/dp_svm_model_eps_0.2.pkl
eps= 0.20 | Acc: 0.5089+/-0.0401 | F1: 0.3675+/-0.2529
        Algorithm 2 fallback fired: effective ε' = 0.1000, Δ = 0.066487
Model successfully exported to output/model/dp_svm_model_eps_0.4.pkl


eps= 0.40 | Acc: 0.5125+/-0.0591 | F1: 0.4282+/-0.1977
        Algorithm 2 fallback fired: effective ε' = 0.2000, Δ = 0.027288
Model successfully exported to output/model/dp_svm_model_eps_0.6.pkl
eps= 0.60 | Acc: 0.5193+/-0.0721 | F1: 0.4741+/-0.1330
        Algorithm 2 fallback fired: effective ε' = 0.3000, Δ = 0.014232
Model successfully exported to output/model/dp_svm_model_eps_0.8.pkl
eps= 0.80 | Acc: 0.5010+/-0.0582 | F1: 0.3883+/-0.2347
Model successfully exported to output/model/dp_svm_model_eps_1.0.pkl
eps= 1.00 | Acc: 0.5188+/-0.0763 | F1: 0.4860+/-0.1204
Model successfully exported to output/model/dp_svm_model_eps_1.2.pkl


eps= 1.20 | Acc: 0.5323+/-0.0753 | F1: 0.5146+/-0.0904
Model successfully exported to output/model/dp_svm_model_eps_1.4.pkl
eps= 1.40 | Acc: 0.5437+/-0.0801 | F1: 0.5332+/-0.0906
Model successfully exported to output/model/dp_svm_model_eps_1.6.pkl
eps= 1.60 | Acc: 0.5573+/-0.0733 | F1: 0.5491+/-0.0820
Model successfully exported to output/model/dp_svm_model_eps_1.8.pkl
eps= 1.80 | Acc: 0.5724+/-0.0729 | F1: 0.5609+/-0.0835


Model successfully exported to output/model/dp_svm_model_eps_2.0.pkl
eps= 2.00 | Acc: 0.5859+/-0.0640 | F1: 0.5757+/-0.0737
Model successfully exported to output/model/dp_svm_model_eps_4.0.pkl
eps= 4.00 | Acc: 0.6833+/-0.0541 | F1: 0.6724+/-0.0541
Model successfully exported to output/model/dp_svm_model_eps_6.0.pkl


eps= 6.00 | Acc: 0.7047+/-0.0453 | F1: 0.6918+/-0.0485
Model successfully exported to output/model/dp_svm_model_eps_8.0.pkl
eps= 8.00 | Acc: 0.7130+/-0.0392 | F1: 0.6997+/-0.0419
Model successfully exported to output/model/dp_svm_model_eps_10.0.pkl
eps=10.00 | Acc: 0.7167+/-0.0380 | F1: 0.7045+/-0.0395


## 6. Save Results


In [5]:
os.makedirs('output', exist_ok=True)
results_df = pd.DataFrame(results)
if baseline_accuracy is not None:
    results_df['accuracy_loss_mean'] = 1 - results_df['accuracy_mean'] / baseline_accuracy
    results_df['accuracy_loss_pct'] = results_df['accuracy_loss_mean'] * 100

results_df.to_csv('output/dp_results.csv', index=False)
with open('output/dp_svm_report.json', 'w') as f:
    json.dump(all_results, f, indent=4)

print("Results saved.")


Results saved.
